In [0]:
# Cell 1: parse NDW situations with a UDF
import xml.etree.ElementTree as ET
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

CATALOG = "city_lens"
BRONZE_TABLE = f"{CATALOG}.bronze.ndw_raw"
SILVER_TABLE = f"{CATALOG}.silver.road_works"
XSI_TYPE = "{http://www.w3.org/2001/XMLSchema-instance}type"

def local(tag):
    return tag.split("}")[-1]

def first_text(root, name):
    for e in root.iter():
        if local(e.tag) == name and e.text:
            return e.text.strip()
    return None

situation_schema = StructType([
    StructField("record_types",   ArrayType(StringType())),
    StructField("severity",       StringType()),
    StructField("version_time",   StringType()),
    StructField("source_name",    StringType()),
    StructField("start_time",     StringType()),
    StructField("end_time",       StringType()),
    StructField("cause_type",     StringType()),
    StructField("delay_band",     StringType()),
    StructField("title",          StringType()),
    StructField("description",    StringType()),
    StructField("lat",            DoubleType()),
    StructField("lon",            DoubleType()),
    StructField("n_points",       IntegerType()),
])

@udf(returnType=situation_schema)
def parse_situation(raw_xml):
    root = ET.fromstring(raw_xml)

    record_types = sorted({
        e.get(XSI_TYPE).split(":")[-1]
        for e in root.iter() if local(e.tag) == "situationRecord" and e.get(XSI_TYPE)
    })

    # Comments: the internalNote reads like a title ("N292 Amentstraat ..."), the rest are details
    title, details = None, []
    for c in root.iter():
        if local(c.tag) != "generalPublicComment":
            continue
        text = first_text(c, "value")
        ctype = first_text(c, "commentType")
        if text:
            if ctype == "internalNote" and title is None:
                title = text
            else:
                details.append(text)

    lats = [float(e.text) for e in root.iter() if local(e.tag) == "latitude" and e.text]
    lons = [float(e.text) for e in root.iter() if local(e.tag) == "longitude" and e.text]
    points = list(zip(lats, lons))
    lat = sum(p[0] for p in points) / len(points) if points else None   # centre of all points
    lon = sum(p[1] for p in points) / len(points) if points else None

    return (
        record_types,
        first_text(root, "overallSeverity"),
        first_text(root, "situationVersionTime"),
        first_text(root, "value"),                 # first value = sourceName (e.g. "Provincie Limburg")
        first_text(root, "overallStartTime"),
        first_text(root, "overallEndTime"),
        first_text(root, "causeType"),
        first_text(root, "delayBand"),
        title or (details[0] if details else None),
        " | ".join(details) if details else None,
        lat, lon, len(points),
    )

latest = spark.sql(f"SELECT max(ingest_date) AS d FROM {BRONZE_TABLE}").first()["d"]

parsed = (
    spark.table(BRONZE_TABLE)
         .where(col("ingest_date") == latest)
         .withColumn("s", parse_situation(col("raw_xml")))
         .select("city_key", "situation_id", col("ingest_ts").alias("updated_ts"), "raw_xml", "s.*")
)
print("Latest bronze date:", latest, "| rows:", parsed.count())
display(parsed.drop("raw_xml").limit(5))

In [0]:
# Cell 2: types, closure flags, newest version per situation, and a profile
latest_version = Window.partitionBy("situation_id").orderBy(col("version_time").desc(), col("updated_ts").desc())

road_works = (
    parsed
    .withColumn("version_time", col("version_time").cast("timestamp"))
    .withColumn("start_time",   col("start_time").cast("timestamp"))
    .withColumn("end_time",     col("end_time").cast("timestamp"))
    .withColumn("is_road_closed", col("raw_xml").rlike("roadClosed|carriagewayClosures"))
    .withColumn("is_lane_closed", col("raw_xml").rlike("laneClosures|narrowLanes"))
    .withColumn("rn", F.row_number().over(latest_version))
    .where(col("rn") == 1)
    .drop("rn", "raw_xml")
)

display(
    road_works.withColumn("record_type", F.explode_outer("record_types"))
              .groupBy("record_type", "is_road_closed", "is_lane_closed")
              .count()
              .orderBy(F.desc("count"))
)

In [0]:
# Cell 3: dashboard-friendly columns, then MERGE into silver.road_works
types = F.concat_ws(",", col("record_types"))

road_works_final = (
    road_works
    .withColumn("kind",
        F.when(types.contains("PublicEvent"), "road_event")
         .when(types.rlike("ConstructionWorks|MaintenanceWorks"), "works")
         .otherwise("traffic_management"))
    .withColumn("closure_level",
        F.when(col("is_road_closed"), "road_closed")
         .when(col("is_lane_closed"), "lane_closed")
         .otherwise("none"))
)

if not spark.catalog.tableExists(SILVER_TABLE):
    road_works_final.limit(0).write.format("delta").saveAsTable(SILVER_TABLE)

road_works_final.createOrReplaceTempView("road_works_updates")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS t
    USING road_works_updates AS s
      ON t.situation_id = s.situation_id
    WHEN MATCHED AND s.version_time >= t.version_time THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"""
    SELECT kind, closure_level,
           count(*)                                       AS situations,
           sum(CASE WHEN current_timestamp() BETWEEN start_time AND end_time THEN 1 ELSE 0 END) AS active_now,
           min(start_time) AS earliest_start,
           max(end_time)   AS latest_end
    FROM {SILVER_TABLE}
    GROUP BY kind, closure_level
    ORDER BY situations DESC
"""))